# Exploratory Data Analysis (EDA) - Customer Master Data

This notebook performs a comprehensive Exploratory Data Analysis (EDA) on the `customer_master.csv` dataset. The goal is to understand customer demographics, segment distribution, geographical presence, and customer acquisition costs (CAC).

### Objectives:
1. Load and inspect the dataset structure.
2. Perform univariate analysis on both numerical and categorical columns.
3. Conduct bivariate and multivariate analysis to identify relationships.
4. Highlight key business insights and recommendations.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load the CSV file into a DataFrame
df = pd.read_csv('customer_master.csv')
print(f"Dataset successfully loaded. Shape: {df.shape}")

ModuleNotFoundError: No module named 'matplotlib'

## 1. Initial Data Inspection

Let's check the structure of our data, check for missing values, and look for duplicates.

In [ ]:
# Display the first 5 rows
print("--- First 5 Rows ---")
display(df.head())

# Display info (data types, non-null counts)
print("\n--- Dataset Information ---")
df.info()

# Check for missing values
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# Check for duplicate rows
print("\n--- Duplicate Rows Count ---")
print(f"Number of duplicate rows: {df.duplicated().sum()}")

## 2. Statistical Summary

Let's look at descriptive statistics for both numerical and categorical variables.

In [ ]:
print("--- Numerical Columns Summary ---")
display(df.describe())

print("\n--- Categorical Columns Summary ---")
display(df.describe(include=['object']))

## 3. Univariate Analysis

We will explore individual features to understand their distributions.

In [ ]:
# Plot distributions of Numerical Features: Age and CAC
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Customer Age
sns.histplot(df['customer_age'], bins=20, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribution of Customer Age', fontsize=14)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Customer Acquisition Cost (CAC)
sns.histplot(df['customer_acquisition_cost'], bins=20, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Distribution of Customer Acquisition Cost (CAC)', fontsize=14)
axes[1].set_xlabel('CAC ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Plot frequencies of Categorical Features: Gender and Customer Segment
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gender distribution
gender_counts = df['gender'].value_counts()
sns.barplot(x=gender_counts.index, y=gender_counts.values, ax=axes[0], hue=gender_counts.index, palette='pastel', legend=False)
axes[0].set_title('Customer Distribution by Gender', fontsize=14)
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
for i, v in enumerate(gender_counts.values):
    axes[0].text(i, v + 100, f"{v} ({v/len(df)*100:.1f}%)", ha='center', fontweight='bold')

# Customer Segment distribution
segment_counts = df['customer_segment'].value_counts()
sns.barplot(x=segment_counts.index, y=segment_counts.values, ax=axes[1], hue=segment_counts.index, palette='Set2', legend=False)
axes[1].set_title('Customer Distribution by Segment', fontsize=14)
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Count')
for i, v in enumerate(segment_counts.values):
    axes[1].text(i, v + 100, f"{v} ({v/len(df)*100:.1f}%)", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Plot geographical distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 Countries
top_countries = df['customer_country'].value_counts().head(10)
sns.barplot(x=top_countries.values, y=top_countries.index, ax=axes[0], hue=top_countries.index, palette='viridis', legend=False)
axes[0].set_title('Top 10 Countries by Customer Count', fontsize=14)
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Country')

# Region distribution
region_counts = df['region'].value_counts()
sns.barplot(x=region_counts.index, y=region_counts.values, ax=axes[1], hue=region_counts.index, palette='rocket', legend=False)
axes[1].set_title('Customer Distribution by Region', fontsize=14)
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Count')
for i, v in enumerate(region_counts.values):
    axes[1].text(i, v + 100, f"{v} ({v/len(df)*100:.1f}%)", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Bivariate & Multivariate Analysis

Let's look at relations between fields to extract meaningful patterns.

In [ ]:
# Box plot of CAC by Customer Segment
plt.figure(figsize=(10, 6))
sns.boxplot(x='customer_segment', y='customer_acquisition_cost', data=df, hue='customer_segment', palette='Set2', legend=False)
plt.title('Customer Acquisition Cost (CAC) by Customer Segment', fontsize=16)
plt.xlabel('Customer Segment')
plt.ylabel('CAC ($)')
plt.show()

# Descriptive statistics for CAC by Segment
print("--- Descriptive Stats for CAC by Segment ---")
display(df.groupby('customer_segment')['customer_acquisition_cost'].describe())

In [ ]:
# Scatter plot between Age and CAC (using alpha for density due to 25k records)
plt.figure(figsize=(10, 6))
sns.scatterplot(x='customer_age', y='customer_acquisition_cost', data=df, alpha=0.1, color='purple')
plt.title('Customer Age vs. Customer Acquisition Cost (CAC)', fontsize=16)
plt.xlabel('Customer Age')
plt.ylabel('CAC ($)')
plt.show()

correlation = df['customer_age'].corr(df['customer_acquisition_cost'])
print(f"Pearson Correlation Coefficient between Customer Age and CAC: {correlation:.4f}")

In [ ]:
# Gender distribution across segments
segment_gender = pd.crosstab(df['customer_segment'], df['gender'], normalize='index') * 100

segment_gender.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='coolwarm')
plt.title('Gender Distribution across Customer Segments (Normalized %)', fontsize=16)
plt.xlabel('Customer Segment')
plt.ylabel('Percentage (%)')
plt.legend(title='Gender', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Average CAC by Country and Segment (Top 5 Countries)
top_5_countries = df['customer_country'].value_counts().head(5).index
df_top_5 = df[df['customer_country'].isin(top_5_countries)]

plt.figure(figsize=(14, 7))
sns.barplot(x='customer_country', y='customer_acquisition_cost', hue='customer_segment', data=df_top_5, palette='Set2')
plt.title('Average Customer Acquisition Cost (CAC) by Country and Segment (Top 5 Countries)', fontsize=16)
plt.xlabel('Country')
plt.ylabel('Average CAC ($)')
plt.legend(title='Customer Segment')
plt.show()

## 5. Key Insights and Business Recommendations

Based on our Exploratory Data Analysis, we can summarize the following findings:

1. **Demographics**:
   - **Age Profile**: The average customer age is **45.9 years**, ranging from 18 to 74 years. The distribution is uniform, indicating a broad and multi-generational customer base.
   - **Gender Split**: The gender split is highly balanced, consisting of ~48.2% Female, ~47.9% Male, and ~3.9% Non-Binary customers.

2. **Customer Acquisition Cost (CAC)**:
   - The average CAC is **$42.16**, with values spanning from $5.01 to $79.99.
   - **No Age Correlation**: There is virtually zero correlation ($r = -0.007$) between customer age and CAC. Age does not impact how expensive it is to acquire a customer.
   - **Uniform CAC across Segments**: Surprisingly, the acquisition cost is highly consistent across all customer segments (Consumer, Business, Premium, VIP). Each segment has a mean CAC around $41 - $43.

3. **Geographical Distributions**:
   - **Market Dominance**: The USA accounts for the vast majority of the customer base (**59.7%**), followed by the UK (**14.8%**), Germany (**8.2%**), Canada (**5.1%**), and Australia (**4.9%**).

### Actionable Business Recommendations:
- **Differentiated Acquisition Budgets**: Since VIP and Premium customers typically generate higher Customer Lifetime Value (LTV), but currently cost the same to acquire as standard Consumer customers (~$42), the business can afford to spend more acquisition budget targeting high-value segments (Premium and VIP) to accelerate growth there.
- **Optimize Consumer Segment Acquisition**: A CAC of ~$42 might be too high for low-tier Consumer accounts. The marketing team should look into lower-cost, organic channels (e.g., referral programs, organic content) to lower the CAC specifically for the Consumer segment.
- **International Expansion**: Assess whether successful acquisition frameworks used in the USA can be adapted to scale up customer acquisition in high-potential markets like Germany, Canada, and Australia.